# Full inference: artifacts + contours + character entities
Self-contained inference from local models and 1–3 ordered page images. No labels, training samples or other experiment caches are read. Run top to bottom in `godfather.venv`.

Magi + Paddle → Paddle-only Gemma artifact filtering → punctuation fallback bubble gate → contour grouping/roles → tail forces SPOKEN. Original fine-tuned character crops, cleanup, entity clustering and member associations remain unchanged. Final transcript OCR correction is optional.

Only this notebook's cache root is read/written. Set `FORCE_ALL=True` for a fresh computation. Runtime helper dependencies are `contour_narrator2.py` and `contour_grouping.py`; their paths/hashes and all model resources are exported. No other notebook is executed. No MLP/CatBoost/DeBERTa is used.

## 1. Imports and offline execution

Import Torch before Paddle, as required by this environment. Model loading uses local files or the existing local Hugging Face cache.

In [24]:
# Set offline mode before importing Hugging Face model loaders.
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
import torch  # Must precede Paddle on this Windows installation.
import paddle
import gc
import copy
import hashlib
import json
import sys
import unicodedata
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
try:
    import ipywidgets as widgets
except ImportError:
    widgets=None
from IPython.display import display
from PIL import Image
from transformers import AutoModel, AutoConfig
from paddleocr import PaddleOCR

print('Python:', sys.executable)
print('Torch:', torch.__version__, '| Paddle:', paddle.__version__)

import importlib.util
import inspect
import types
from scipy.optimize import linear_sum_assignment

# Load a local Python file under a unique module name: avoid collisions with other config/model files.
def load_local_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

import math
import re
from difflib import SequenceMatcher
from io import BytesIO
from IPython.display import Image as DisplayImage, clear_output
import cv2


Python: c:\Users\ESHAAN\BYTE\godfather.venv\Scripts\python.exe
Torch: 2.14.0+cu126 | Paddle: 3.2.0


## 2. Local resources and input settings
Configure models, character checkpoint, helpers and either TaskData or INPUT_IMAGES. No training labels are needed. FORCE_ALL controls only this notebook’s own caches.

In [25]:
# Local resource paths: these are the only model/checkpoint resources required.
MAGI_ROOT=Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi')
PIPELINE_ROOT=MAGI_ROOT/'paddleOCR-Magi'
# Model source: https://huggingface.co/ragavsachdeva/magiv2
MODEL_PATH=Path('C:/MODELS/magiv2')
# Model source: https://huggingface.co/google/gemma-3-4b-it
GEMMA_MODEL_PATH=Path('C:/MODELS/gemma-3-4b-it')
# Bubble checkpoint source: https://huggingface.co/PSImera/manga_bubbles_detect
BUBBLE_MODEL=Path('C:/MODELS/manga_bubbles/bubbles_detect.pt')
# Our fine-tuned checkpoint is not published yet; base: https://huggingface.co/ragavsachdeva/magiv2
CHECKPOINT_PATH=MAGI_ROOT/'FINETUNING-EXPERMENTS/fineTuningMagiCharSimHeads2/magi_crop_last_2_blocks.pt'
# Model source: https://huggingface.co/PaddlePaddle/PP-OCRv5_server_det
PADDLE_DET_PATH=Path.home()/'.paddlex/official_models/PP-OCRv5_server_det'
# Model source: https://huggingface.co/PaddlePaddle/en_PP-OCRv5_mobile_rec
PADDLE_REC_PATH=Path.home()/'.paddlex/official_models/en_PP-OCRv5_mobile_rec'

# Use TaskData, or set INPUT_IMAGES to an ordered list of one to three image paths.
SPLIT='development'
SEQUENCE_ID='seq_2fae297a18af2a57'
PAGE_NUMBER=None  # Dataset mode only: None selects the complete sequence.
INPUT_IMAGES=None
# INPUT_IMAGES=[Path('C:/path/page1.png'),Path('C:/path/page2.png')]
CACHE_ROOT=PIPELINE_ROOT/'cache_artifact_contour_full_inference'
FORCE_ALL=False
CLEAN_TRANSCRIPT=True
SAVE_ANNOTATED_PAGES=False

# Existing character pipeline settings/checkpoint stay unchanged.
ENTITY_THRESHOLD=0.82
THRESHOLD_CHECKPOINT_EPOCH=20
CHARACTER_CLEANUP_SETTINGS=dict(min_iou=.90,min_cosine=.95,algorithm='contained_direct_outer_match_v1')
ASSOCIATION_HEAD_PATH=None  # Original Magi association head by default.
MODEL_REVISION='local-magiv2-1'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
LANGUAGE='en'
PADDLE_SETTINGS=dict(lang=LANGUAGE,device='gpu:0',ocr_version='PP-OCRv5',
    use_doc_orientation_classify=False,use_doc_unwarping=False,use_textline_orientation=False)
PADDLE_LOCAL_SETTINGS=dict(text_detection_model_name='PP-OCRv5_server_det',
    text_recognition_model_name='en_PP-OCRv5_mobile_rec',
    text_detection_model_dir=str(PADDLE_DET_PATH),text_recognition_model_dir=str(PADDLE_REC_PATH))
DETECTION=dict(imgsz=1024,confidence=.25)
CONTOUR_MODULE_PATH=PIPELINE_ROOT/'contour_narrator2.py'
CONTOUR_GROUPING_PATH=PIPELINE_ROOT/'contour_grouping.py'
contour_model=load_local_module('standalone_artifact_contour_rules',CONTOUR_MODULE_PATH)
CONTOUR_SETTINGS=dict(contour_model.CHECK)
REGION_SETTINGS=dict(contour_model.REGION)
FAINT_REGION_SETTINGS=dict(contour_model.FAINT_REGION)
GROUP_FAINT_SETTINGS=dict(contour_model.contours.FAINT)
contour_model.CHECK=CONTOUR_SETTINGS;contour_model.REGION=REGION_SETTINGS
contour_model.FAINT_REGION=FAINT_REGION_SETTINGS;contour_model.contours.FAINT=GROUP_FAINT_SETTINGS
STAGES=['magi_raw','embeddings','paddle_ocr','ocr_selection','artifact_labels',
    'artifact_filter','character_cleanup','entities','text_entities','contour_bubbles',
    'contour_roles','contour_speakers','gemma_transcript']
FORCE={name:FORCE_ALL for name in STAGES}
STAGE_VERSIONS={name:1 for name in STAGES}


## 3. Load one to three images
Use ordered explicit paths for standalone input, or dataset sequence/page selection. No other caches are searched.

In [26]:
SPLIT=str(SPLIT).strip();SEQUENCE_ID=str(SEQUENCE_ID).strip()
assert SEQUENCE_ID and Path(SEQUENCE_ID).name==SEQUENCE_ID,'Use a single sequence name, not a path'
if INPUT_IMAGES is None:
    SEQUENCE_ROOT=MAGI_ROOT/'TaskData'/SPLIT/'images'/SEQUENCE_ID
    assert SEQUENCE_ROOT.is_dir(),f'Missing image directory: {SEQUENCE_ROOT}'
    sequence_files=sorted(p for p in SEQUENCE_ROOT.iterdir() if p.suffix.lower()=='.png' and p.stem.isdigit())
    PAGE_FILES=sequence_files if PAGE_NUMBER is None else [p for p in sequence_files if int(p.stem)==int(PAGE_NUMBER)]
    PAGE_LABELS=[int(p.stem) for p in PAGE_FILES]
else:
    PAGE_FILES=[Path(p).expanduser().resolve() for p in INPUT_IMAGES]
    sequence_files=list(PAGE_FILES)
    PAGE_LABELS=list(range(1,len(PAGE_FILES)+1))
assert 1<=len(PAGE_FILES)<=3,'Supply one to three ordered page images'
assert len({str(p.resolve()) for p in PAGE_FILES})==len(PAGE_FILES),'Repeated input image path'
assert len(PAGE_LABELS)==len(set(PAGE_LABELS)),'Repeated page ID'
for path in PAGE_FILES:assert path.is_file(),f'Missing input image: {path}'
for path in [MODEL_PATH,GEMMA_MODEL_PATH,PADDLE_DET_PATH,PADDLE_REC_PATH]:
    assert path.is_dir(),f'Missing local model directory: {path}'
for path in [CHECKPOINT_PATH,BUBBLE_MODEL,CONTOUR_MODULE_PATH,CONTOUR_GROUPING_PATH]:
    assert path.is_file(),f'Missing local resource: {path}'
if ASSOCIATION_HEAD_PATH is not None:
    ASSOCIATION_HEAD_PATH=Path(ASSOCIATION_HEAD_PATH);assert ASSOCIATION_HEAD_PATH.is_file()
chapter_pages=[];crop_pages=[]
for path in PAGE_FILES:
    with Image.open(path) as image:
        chapter_pages.append(np.asarray(image.convert('L').convert('RGB')))
        crop_pages.append(np.asarray(image.convert('RGB')))
# A deterministic scope separates different selected image lists; all reuse stays inside CACHE_ROOT.
scope=hashlib.sha256(json.dumps([str(p.resolve()) for p in PAGE_FILES]).encode()).hexdigest()[:16]
CACHE_DIR=CACHE_ROOT/SEQUENCE_ID/scope
OUTPUT_DIR=CACHE_DIR/'outputs'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Images:',[str(p) for p in PAGE_FILES])
print('Own cache only:',CACHE_DIR)


Images: ['C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\TaskData\\development\\images\\seq_2fae297a18af2a57\\01.png', 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\TaskData\\development\\images\\seq_2fae297a18af2a57\\02.png', 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\TaskData\\development\\images\\seq_2fae297a18af2a57\\03.png']
Own cache only: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_artifact_contour_full_inference\seq_2fae297a18af2a57\a922ef5f36d9a3aa


## 4. Own-cache helpers and dependency manifest
Read only CACHE_DIR. Each stage validates its input/settings signature and payload checksum. Misses compute and save here. Model/checkpoint resources and helper paths/hashes are exported for packaging.

In [27]:
def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


def digest(data):
    text = json.dumps(json_ready(data), sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(data), ensure_ascii=True,
                                    indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def model_fingerprint():
    files = []
    for path in sorted(MODEL_PATH.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.py', '.safetensors', '.bin', '.pt', '.pth'}:
            stat = path.stat()
            files.append(dict(name=str(path.relative_to(MODEL_PATH)), size=stat.st_size,
                              mtime_ns=stat.st_mtime_ns,
                              sha256=file_hash(path) if path.suffix in {'.json', '.py'} else None))
    return dict(path=str(MODEL_PATH), revision=MODEL_REVISION, files=files)


PACKAGES = {name: version(name) for name in ['torch', 'torchvision', 'transformers',
            'paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}
PAGE_METADATA = [dict(name=p.name, sha256=file_hash(p), width=image.shape[1], height=image.shape[0])
                 for p, image in zip(PAGE_FILES, chapter_pages)]
INPUTS = dict(sequence_id=SEQUENCE_ID, split=SPLIT, pages=PAGE_METADATA,
              preprocessing='grayscale_then_RGB')
MODEL_INFO = model_fingerprint()
if ASSOCIATION_HEAD_PATH is not None:
    MODEL_INFO['association_head_sha256'] = file_hash(ASSOCIATION_HEAD_PATH)
# Our trained two-block checkpoint; original base: https://huggingface.co/ragavsachdeva/magiv2
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert checkpoint['version'] == 2 and checkpoint['settings']['trained_blocks'] == 2
assert checkpoint['best_epoch'] == THRESHOLD_CHECKPOINT_EPOCH, (
    'Checkpoint epoch changed. Set ENTITY_THRESHOLD and THRESHOLD_CHECKPOINT_EPOCH from its validation clustering run.')
for name, expected in checkpoint['base_model_hashes'].items():
    assert file_hash(MODEL_PATH / name) == expected, f'Original model changed: {name}'
CHECKPOINT_INFO = dict(path=str(CHECKPOINT_PATH), sha256=file_hash(CHECKPOINT_PATH),
                       epoch=checkpoint['best_epoch'], trained_blocks=2)
manifest = dict(schema_version=1, inputs=INPUTS, model=MODEL_INFO, packages=PACKAGES,
                crop_encoder=CHECKPOINT_INFO, stages={})
print('Crop checkpoint epoch:', checkpoint['best_epoch'], '| clustering threshold:', ENTITY_THRESHOLD)
MANIFEST_PATH = CACHE_DIR / 'manifest.json'
if MANIFEST_PATH.exists():
    try:
        old_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        if old_manifest.get('inputs') == INPUTS:
            manifest['stages'] = old_manifest.get('stages', {})
    except (ValueError, OSError):
        pass


def stage_ref(name):
    entry = manifest['stages'][name]
    return dict(signature=entry['signature'], data_hash=entry['data_hash'])


def cached_stage(name, settings, compute):
    signature = digest(dict(inputs=INPUTS, algorithm_version=STAGE_VERSIONS[name], settings=settings))
    path = CACHE_DIR / f'{name}.json'
    payload = None
    if path.exists() and not FORCE[name]:
        try:
            candidate = json.loads(path.read_text(encoding='utf-8'))
            if candidate['signature'] == signature and digest(candidate['data']) == candidate['data_hash']:
                payload = candidate
        except (ValueError, KeyError, OSError):
            pass
    if payload is None:
        print(f'RUN   {name}')
        data = json_ready(compute())
        payload = dict(signature=signature, data_hash=digest(data),
                       created_at=datetime.now(timezone.utc).isoformat(), data=data)
        write_json(path, payload)
    else:
        print(f'CACHE {name}')
    manifest['stages'][name] = dict(file=path.name, signature=signature,
                                  data_hash=payload['data_hash'], settings=settings,
                                  algorithm_version=STAGE_VERSIONS[name], created_at=payload['created_at'])
    write_json(MANIFEST_PATH, manifest)
    return payload['data']

def function_digest(function):
    # Function source is added to new stages' cache keys, so notebook edits invalidate their results.
    # Ignore notebook execution filenames/line numbers; they change across kernel restarts.
    def describe(code):
        constants = [describe(value) if isinstance(value, types.CodeType) else repr(value)
                     for value in code.co_consts]
        return dict(bytecode=code.co_code.hex(), names=code.co_names,
                    variables=code.co_varnames, constants=constants)
    return digest(describe(inspect.unwrap(function).__code__))

DEPENDENCIES=[dict(path=str(path.resolve()),sha256=file_hash(path),role=role)
    for path,role in [(CONTOUR_MODULE_PATH,'background regions and SPOKEN/NARRATED'),
                      (CONTOUR_GROUPING_PATH,'individual boundaries, faint retry and overlap groups')]]
GEMMA_IDENTITY=dict(path=str(GEMMA_MODEL_PATH),files={
    str(p.relative_to(GEMMA_MODEL_PATH)):dict(size=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns,
        sha256=file_hash(p) if p.suffix=='.json' else None)
    for p in sorted(GEMMA_MODEL_PATH.rglob('*')) if p.is_file() and p.suffix in {'.json','.safetensors','.bin','.model'}})
RESOURCE_MANIFEST=dict(helpers=DEPENDENCIES,
    models=dict(magi=MODEL_INFO,character_checkpoint=CHECKPOINT_INFO,gemma=GEMMA_IDENTITY,
        bubble=dict(path=str(BUBBLE_MODEL),sha256=file_hash(BUBBLE_MODEL)),paddle=PADDLE_LOCAL_SETTINGS),
    cache_policy='Read/write CACHE_ROOT only; no external cache discovery',
    input_images=[str(p.resolve()) for p in PAGE_FILES])
write_json(OUTPUT_DIR/'dependencies.json',RESOURCE_MANIFEST)


Crop checkpoint epoch: 20 | clustering threshold: 0.82


## 5. Magi and character embedding functions

Original detections/associations are produced before patching the final two character crop-ViT blocks. The optional association checkpoint changes only the text-character matching head.

In [28]:
magi_model = None


def get_magi():
    global magi_model
    if magi_model is None:
        assert torch.cuda.is_available(), 'Torch CUDA unavailable'
        # Normal tensors allow the later checkpoint copy; only predictions use inference mode.
        with torch.inference_mode(False):
            # Base model: https://huggingface.co/ragavsachdeva/magiv2
            config = AutoConfig.from_pretrained(str(MODEL_PATH), trust_remote_code=True, local_files_only=True)
            # The full checkpoint already contains the backbone: avoid timm downloading it again.
            config.detection_model_config.use_pretrained_backbone = False
            config.disable_ocr = False
            config.disable_crop_embeddings = False
            magi_model, info = AutoModel.from_pretrained(str(MODEL_PATH), config=config,
                trust_remote_code=True, local_files_only=True, output_loading_info=True)
            assert not info['missing_keys'] and not info.get('mismatched_keys'), info
            magi_model = magi_model.cuda().eval()
            magi_model.requires_grad_(False)
            if ASSOCIATION_HEAD_PATH is not None:
                association = torch.load(ASSOCIATION_HEAD_PATH, map_location='cpu', weights_only=True)
                magi_model.text_character_matching_head.load_state_dict(association['state_dict'], strict=True)
    return magi_model


def predict_magi():
    model = get_magi()
    with torch.inference_mode():
        return model.do_chapter_wide_prediction(
            chapter_pages, {'images': [], 'names': []}, use_tqdm=True, do_ocr=True)


def predict_embeddings():
    boxes = [result['characters'] for result in magi_raw_results]
    if not any(boxes):
        return [[] for _ in boxes]
    model = get_magi()
    last = len(model.crop_embedding_model.encoder.layer) - 1
    assert last >= 1
    prefixes = (f'crop_embedding_model.encoder.layer.{last-1}.',
                f'crop_embedding_model.encoder.layer.{last}.')
    parameters = dict(model.named_parameters())
    expected_names = {name for name in parameters if name.startswith(prefixes)}
    assert tuple(checkpoint['trainable_prefixes']) == prefixes
    assert set(checkpoint['state_dict']) == expected_names
    # Patch only the two crop blocks after original Magi prediction is complete.
    with torch.no_grad():
        for name, tensor in checkpoint['state_dict'].items():
            assert tensor.shape == parameters[name].shape
            parameters[name].copy_(tensor.to(parameters[name].device))
    model.eval()
    with torch.inference_mode():
        embeddings = model.predict_crop_embeddings(crop_pages, boxes, mask_ratio=0.0, batch_size=32)
        return [torch.nn.functional.normalize(x, p=2, dim=1) if len(x) else x for x in embeddings]


def validate_magi(results):
    assert len(results) == len(chapter_pages), 'Magi page count mismatch'
    for result in results:
        assert 'tails' in result and 'text_tail_associations' in result,'Magi must return tail detections/links'
        assert len(result['texts']) == len(result['ocr']), 'Text/OCR count mismatch'
        assert len(result['characters']) == len(result['character_cluster_labels']), 'Character/cluster count mismatch'
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']), 'Invalid text association index'
            assert 0 <= char_index < len(result['characters']), 'Invalid character association index'


## 6. Create or reuse this notebook’s Magi/character caches
Original detections/associations first, then patch only the existing fine-tuned two character crop blocks. Models are released before OCR/Gemma.

In [29]:
try:
    magi_raw_results = cached_stage('magi_raw',
        dict(model=MODEL_INFO, torch=PACKAGES['torch'], torchvision=PACKAGES['torchvision'],
             transformers=PACKAGES['transformers'], do_ocr=True), predict_magi)
    validate_magi(magi_raw_results)
    embeddings_per_page = cached_stage('embeddings',
        dict(magi=stage_ref('magi_raw'), model=MODEL_INFO, checkpoint=CHECKPOINT_INFO,
             crop_preprocessing='original_RGB', mask_ratio=0.0, batch_size=32, normalization='L2',
             torch=PACKAGES['torch'], transformers=PACKAGES['transformers']), predict_embeddings)
finally:
    magi_model = None
    gc.collect()
    torch.cuda.empty_cache()

assert len(embeddings_per_page) == len(magi_raw_results)
for result, embeddings in zip(magi_raw_results, embeddings_per_page):
    assert len(embeddings) == len(result['characters']), 'Character/embedding count mismatch'
    assert np.isfinite(np.asarray(embeddings)).all(), 'Nonfinite embeddings'
per_page_results = copy.deepcopy(magi_raw_results)
print('Text counts:', [len(r['texts']) for r in per_page_results])


CACHE magi_raw
CACHE embeddings
Text counts: [11, 9, 8]


## 7. Paddle OCR functions

Recognize each original text crop. Invalid crops and per-box errors keep placeholders, allowing Magi OCR fallback without shifting text indices.

In [30]:
def crop_box(image, box):
    values = np.asarray(box, dtype=float)
    if values.shape != (4,) or not np.isfinite(values).all():
        return None, None
    height, width = image.shape[:2]
    x1, y1, x2, y2 = values
    clipped = [int(np.clip(np.floor(x1), 0, width)), int(np.clip(np.floor(y1), 0, height)),
               int(np.clip(np.ceil(x2), 0, width)), int(np.clip(np.ceil(y2), 0, height))]
    x1, y1, x2, y2 = clipped
    if x2 <= x1 or y2 <= y1:
        return None, clipped
    return np.ascontiguousarray(image[y1:y2, x1:x2]), clipped


def predict_paddle():
    output = [[] for _ in chapter_pages]
    if not any(result['texts'] for result in magi_raw_results):
        return output
    assert paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0, 'Paddle CUDA unavailable'
    assert PADDLE_DET_PATH.is_dir() and PADDLE_REC_PATH.is_dir(), 'Set the local Paddle model paths in cell 2.'
    # Explicit local directories prevent downloading detector/recognizer models.
    # Detector: https://huggingface.co/PaddlePaddle/PP-OCRv5_server_det
    # Recognizer: https://huggingface.co/PaddlePaddle/en_PP-OCRv5_mobile_rec
    engine = PaddleOCR(**PADDLE_SETTINGS, **PADDLE_LOCAL_SETTINGS)
    try:
        for page_index, (image, result) in enumerate(zip(chapter_pages, magi_raw_results)):
            for text_index, box in enumerate(result['texts']):
                crop, clipped = crop_box(image, box)
                record = dict(page_index=page_index, text_index=text_index, text_id=f'T{text_index+1}',
                              box=box, crop_box=clipped, text='', lines=[], status='invalid_crop', error=None)
                if crop is not None:
                    try:
                        for prediction in engine.predict(input=crop):
                            texts, scores = prediction['rec_texts'], prediction['rec_scores']
                            assert len(texts) == len(scores), 'Paddle text/score count mismatch'
                            for text, score in zip(texts, scores):
                                record['lines'].append(dict(text=str(text), confidence=float(score)))
                        record['text'] = ' '.join(line['text'] for line in record['lines'])
                        record['status'] = 'text' if record['text'].strip() else 'empty'
                    except Exception as error:
                        record.update(text='', lines=[], status='error', error=str(error))
                        print(f'PG{page_index+1}:T{text_index+1} OCR error: {error}')
                output[page_index].append(record)
    finally:
        del engine
        gc.collect()
        paddle.device.cuda.empty_cache()
    return output


## 8. Paddle OCR cache
Each original Magi text crop is read locally. Empty/error output keeps the original index for audit and punctuation fallback.

In [31]:
paddle_ocr_results = cached_stage('paddle_ocr',
    dict(magi=stage_ref('magi_raw'), paddle=PADDLE_SETTINGS, local_models=PADDLE_LOCAL_SETTINGS,
         packages={k: PACKAGES[k] for k in ['paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}),
    predict_paddle)
print('Paddle records:', sum(len(page) for page in paddle_ocr_results))
print('Cached OCR errors:', sum(r['status'] == 'error' for page in paddle_ocr_results for r in page))


CACHE paddle_ocr
Paddle records: 28
Cached OCR errors: 0


## 9. Paddle-only Gemma artifact classifier
Prompt/examples are embedded below. No agreement threshold or separate watermark/SFX rule deletes boxes first. Punctuation-only Paddle is Text; empty Paddle uses Magi punctuation or is Artifact. Unexpected answers are Review and retained.

In [32]:
SYSTEM_PROMPT = 'Classify the Paddle OCR text of one manga text box. Return exactly Text or Artifact. Text includes dialogue, narration, names, Japanese words/names, unfinished sentences, multiple sentences and multiple lines. Profanity and figurative language are allowed. Integers, integer-only text and numbers within meaningful dialogue are Text. Punctuation-only expressions are Text. Vocal reactions, screams, disgust, hesitation and stretched spoken words such as AAAAHH!!!, NOO!! and EWWW! are Text. Distinguish a human vocal reaction from a written environmental or action sound effect: Whoosh, BAM, CLANG and similar sound effects are Artifact. Artifact also includes nonsensical OCR noise, scan-group watermarks, website links and isolated code-like strings such as 2-B, 3-2 and V45S. Single Text Char are Artifacts Do not reject an unfamiliar name or a meaningful sentence simply because it contains a number or identifier. If uncertain, prefer Text. Do not correct, rewrite or explain the text.'

# One Paddle OCR input and one label per example.
EXAMPLES = [
    ('GAAAAAAHH!!!', 'Text'),
    ('AAAAHH!!!', 'Text'),
    ('AAAH!', 'Text'),
    ('NOO!!', 'Text'),
    ('EWWW!', 'Text'),
    ('UGH... THAT SMELLS AWFUL!', 'Text'),
    ('HUH?!', 'Text'),
    ('OH...', 'Text'),
    ('UM... I THINK SO.', 'Text'),
    ('HAHAHA!', 'Text'),
    ('OW! THAT HURTS!', 'Text'),
    ('YAAAY!!', 'Text'),
    ('I SAID TO NOT DO THAT.', 'Text'),
    ('But our characters were unaware', 'Text'),
    ('NANIKAWA', 'Text'),
    ('SHIROGANE, WAIT!', 'Text'),
    ("IT'S A GODDAMN MIRACLE, THE MONEY JUST VANISHED INTO THIN AIR.", 'Text'),
    ('IF SHE SUCCEEDS IN LURING OUT ONE PERSON IN THIS PLACE,', 'Text'),
    ('WAIT! WHERE DID EVERYONE GO? I WAS JUST HERE.', 'Text'),
    ('I TRUSTED YOU. NOW TELL ME WHAT HAPPENED.', 'Text'),
    ('WE HAVE TO LEAVE.\nTHE TRAIN IS ALREADY HERE!', 'Text'),
    ('Among the six who possessed the aptitude,\nSaiko-chan had the highest aptitude.', 'Text'),
    ('123', 'Text'),
    ('42', 'Text'),
    ('0', 'Text'),
    ('I HAVE 2 QUESTIONS.', 'Text'),
    ('WE HAVE 3 MINUTES LEFT!', 'Text'),
    ('HE PAID 500 YEN FOR IT.', 'Text'),
    ('I AM IN CLASS 2-B.', 'Text'),
    ('THE SCORE IS 3-2. WE CAN STILL WIN!', 'Text'),
    ('...', 'Text'),
    ('!!!!', 'Text'),
    ('...!!!', 'Text'),
    ('?!', 'Text'),
    ('もしもし', 'Artifact'),
    ('しし', 'Artifact'),
    ('も', 'Artifact'),
    ('a ~', 'Artifact'),
    ('~~~~~ ', 'Artifact'),
    ('~~~~~ ', 'Artifact'),
    ('Whoosh!!', 'Artifact'),
    ('VOOSH', 'Artifact'),
    ('KIRK', 'Artifact'),
    ('bEE', 'Artifact'),
    ('WFAG', 'Artifact'),
    ('VOOOSH', 'Artifact'),
    ('O', 'Artifact'),
    ('K', 'Artifact'),
    ('AKK', 'Artifact'),
    ('HUPP', 'Artifact'),
    ('CRASHHH', 'Artifact'),
    ('SHHH', 'Artifact'),
    ('KRASHHH', 'Artifact'),
    ('BAM', 'Artifact'),
    ('CLANG!', 'Artifact'),
    ('THUD', 'Artifact'),
    ('CRASH!!', 'Artifact'),
    ('RUSTLE RUSTLE', 'Artifact'),
    ('CLICK', 'Artifact'),
    ('SPLASH!', 'Artifact'),
    ('SFX', 'Artifact'),
    ('SKX', 'Artifact'),
    ('www.google.com', 'Artifact'),
    ('https://mangastream.com/read/123', 'Artifact'),
    ('MANGA STREAM DO NOT MIRROR', 'Artifact'),
    ('M4NGA STRE4M', 'Artifact'),
    ('DO NOT REDISTRIBUTE', 'Artifact'),
    ('oo', 'Artifact'),
    ('Chapter 32', 'Artifact'),
    ('Chapter 12', 'Artifact'),
    ('Chapter Over', 'Artifact'),
    ('Chapter End', 'Artifact'),
    ('To be Continued', 'Artifact'),
    ('Chapter', 'Artifact'),
    ('XZQWZXQ', 'Artifact'),
    ('2-B', 'Artifact'),
    ('3-2', 'Artifact'),
    ('V45S', 'Artifact'),
]


def punctuation_only(text):
    text=str(text or '').strip()
    return bool(text) and all(c.isspace() or unicodedata.category(c).startswith('P') for c in text)


def direct_decision(paddle,magi=''):
    # Only punctuation may be recovered from Magi when Paddle finds no text.
    if not str(paddle or '').strip():
        if punctuation_only(magi):return 'Text','Magi punctuation fallback: Paddle OCR empty'
        return 'Artifact','Paddle OCR empty; no Magi punctuation fallback'
    if punctuation_only(paddle):return 'Text','punctuation-only override'
    return None,None


def paddle_prompt(paddle):
    return json.dumps({'Paddle OCR':str(paddle or '')},ensure_ascii=False)


def classify_paddle(paddle,model,tokenizer):
    messages=[dict(role='system',content=SYSTEM_PROMPT)]
    for example,label in EXAMPLES:
        messages.extend([dict(role='user',content=paddle_prompt(example)),dict(role='assistant',content=label)])
    messages.append(dict(role='user',content=paddle_prompt(paddle)))
    inputs=tokenizer.apply_chat_template(messages,add_generation_prompt=True,
        tokenize=True,return_dict=True,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output=model.generate(**inputs,do_sample=False,max_new_tokens=8)
    answer=tokenizer.decode(output[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
    # Unexpected model output stays visible as Review instead of causing deletion.
    label={'text':'Text','artifact':'Artifact'}.get(answer.casefold(),'Review')
    return dict(label=label,raw_answer=answer,reason='Gemma classification' if label!='Review' else 'unexpected Gemma answer')


# Keep original indices/boxes for character associations; choose Paddle or Magi punctuation only.
def select_sequence_ocr():
    selections=[]
    for page_index,(raw,paddle_records) in enumerate(zip(magi_raw_results,paddle_ocr_results)):
        assert len(raw['texts'])==len(raw['ocr'])==len(paddle_records)
        page=[]
        for text_index,(box,magi_text,record) in enumerate(zip(raw['texts'],raw['ocr'],paddle_records)):
            assert record['box']==box and record['text_index']==text_index and record['page_index']==page_index
            paddle_text=str(record['text'] or '');magi_text=str(magi_text or '')
            text=paddle_text.strip() if paddle_text.strip() else magi_text.strip() if punctuation_only(magi_text) else ''
            source='paddle' if paddle_text.strip() else 'magi_punctuation' if text else None
            page.append(dict(page_index=page_index,text_index=text_index,text_id=f'T{text_index+1}',box=box,
                magi_text=magi_text,paddle_text=paddle_text,selected_text=text,selected_source=source,
                reason='paddle' if source=='paddle' else 'Magi punctuation fallback' if text else 'empty Paddle; no punctuation fallback',
                needs_review=False,disagreement=magi_text.casefold()!=paddle_text.casefold(),
                paddle_status=record['status'],paddle_error=record['error'],paddle_lines=record['lines']))
        selections.append(page)
    return selections


def classify_artifact_sequence():
    by_text={};pending=[]
    for page in ocr_selection:
        for record in page:
            label,reason=direct_decision(record['paddle_text'],record['magi_text'])
            key=record['paddle_text']
            if label is None and key not in pending:pending.append(key)
    if pending:
        from transformers import AutoTokenizer,Gemma3ForConditionalGeneration,BitsAndBytesConfig
        assert torch.cuda.is_available(),'Gemma requires the CUDA godfather.venv kernel'
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        # Model/tokenizer: https://huggingface.co/google/gemma-3-4b-it
        tokenizer=AutoTokenizer.from_pretrained(str(GEMMA_MODEL_PATH),local_files_only=True)
        model=Gemma3ForConditionalGeneration.from_pretrained(str(GEMMA_MODEL_PATH),local_files_only=True,
            device_map='auto',dtype=dtype,quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=dtype)).eval()
        try:
            for i,text in enumerate(pending,1):
                by_text[text]=classify_paddle(text,model,tokenizer)
                print('Artifact classification',i,'/',len(pending),by_text[text]['label'])
        finally:
            del model,tokenizer;gc.collect();torch.cuda.empty_cache()
    pages=[]
    for records in ocr_selection:
        page=[]
        for record in records:
            label,reason=direct_decision(record['paddle_text'],record['magi_text'])
            decision=dict(label=label,reason=reason,raw_answer='') if label else by_text[record['paddle_text']]
            a=''.join(re.findall('[a-z0-9]',record['magi_text'].casefold()))
            b=''.join(re.findall('[a-z0-9]',record['paddle_text'].casefold()))
            page.append(dict(gemma_label=decision['label'],artifact_label=decision['label'],
                gemma_raw_answer=decision['raw_answer'],artifact_reason=decision['reason'],
                ocr_agreement=SequenceMatcher(None,a,b).ratio() if a and b else None,
                delete_ocr=decision['label']=='Artifact',
                deletion_reason=decision['reason'] if decision['label']=='Artifact' else '',
                punctuation_bubble_checked=False,punctuation_bubble_found=None))
        pages.append(page)
    return pages


## 10. Run Gemma and preserve original box order
Only Paddle OCR is sent to Gemma. Original indices and Magi/Paddle readings remain available for character associations and audit. Agreement is displayed only. Gemma is released before later stages.

In [33]:
ocr_selection=cached_stage('ocr_selection',dict(magi=stage_ref('magi_raw'),paddle=stage_ref('paddle_ocr'),
    algorithm=function_digest(select_sequence_ocr),punctuation=function_digest(punctuation_only)),select_sequence_ocr)
artifact_initial=cached_stage('artifact_labels',dict(ocr=stage_ref('ocr_selection'),model=GEMMA_IDENTITY,
    system=SYSTEM_PROMPT,examples=EXAMPLES,input='paddle_only',quantization='nf4',max_new_tokens=8,
    transformers=version('transformers'),bitsandbytes=version('bitsandbytes'),
    algorithm=function_digest(classify_artifact_sequence),classifier=function_digest(classify_paddle),
    direct=function_digest(direct_decision)),classify_artifact_sequence)
# This adapter preserves the existing character/speaker mapping contract.
ocr_agreement=copy.deepcopy(artifact_initial)  # Agreement is diagnostic only; it never deletes here.
per_page_results=copy.deepcopy(magi_raw_results)
for page_index,(result,decisions,labels) in enumerate(zip(per_page_results,ocr_selection,artifact_initial)):
    result['magi_ocr']=list(result['ocr'])
    result['ocr']=[d['selected_text'] for d in decisions]
    for decision,label in zip(decisions,labels):decision['needs_review']=label['gemma_label']=='Review'
display(pd.DataFrame([dict(page=PAGE_LABELS[p],box_id=f"PG{PAGE_LABELS[p]}:T{i+1}",
    paddle=d['paddle_text'],magi=d['magi_text'],text=d['selected_text'],**artifact_initial[p][i])
    for p,page in enumerate(ocr_selection) for i,d in enumerate(page)]))


CACHE ocr_selection
CACHE artifact_labels


,page,box_id,paddle,magi,text,gemma_label,artifact_label,gemma_raw_answer,artifact_reason,ocr_agreement,delete_ocr,deletion_reason,punctuation_bubble_checked,punctuation_bubble_found
0,1,PG1:T1,MANGA STREAM DO NOT MIRROR,"MANGA STREAN, DO NOTMIROR",MANGA STREAM DO NOT MIRROR,Artifact,Artifact,Artifact,Gemma classification,0.930233,True,Gemma classification,False,None
1,1,PG1:T2,GOT YOUR TOMATO DEVIL RIGHT HERE.,Got your tomato devil right here.,GOT YOUR TOMATO DEVIL RIGHT HERE.,Text,Text,Text,Gemma classification,1.000000,False,,False,None
2,1,PG1:T3,"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.","It revolves from its seeds, so you should burn...","IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",Text,Text,Text,Gemma classification,0.961039,False,,False,None
3,1,PG1:T4,"GOOD WORK, DENJI.","Good work, Denji.","GOOD WORK, DENJI.",Text,Text,Text,Gemma classification,1.000000,False,,False,None
4,1,PG1:T5,THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON T...,This guy's corpse'll fetch a pretty penny on t...,THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON T...,Text,Text,Text,Gemma classification,1.000000,False,,False,None
5,1,PG1:T6,THE REWARD'S 400 GRAND.,The reward's 400 grand.,THE REWARD'S 400 GRAND.,Text,Text,Text,Gemma classification,1.000000,False,,False,None
6,1,PG1:T7,,Though you look at the bottom-of-the-the-the-a...,,Artifact,Artifact,,Paddle OCR empty; no Magi punctuation fallback,NaN,True,Paddle OCR empty; no Magi punctuation fallback,False,None
7,1,PG1:T8,THANKS !!,Thanks !!,THANKS !!,Text,Text,Text,Gemma classification,1.000000,False,,False,None
8,1,PG1:T9,I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,I'm taking out 170 grand to cover your loan an...,I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,Text,Text,Text,Gemma classification,0.962264,False,,False,None
9,1,PG1:T10,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,And then there's the commission for referring ...,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,Text,Text,Text,Gemma classification,0.982759,False,,False,None


## 11. Remove duplicate character appearances

Reuse the existing containment + IoU + embedding-similarity rule. Redirect text-character links, remap character indices, and leave text indices unchanged.

In [34]:
def remove_duplicate_characters(results, embeddings, min_iou=0.90, min_cosine=0.95):
    """Remove contained, visually similar duplicates; preserve all text indices."""
    assert 0 <= min_iou <= 1 and -1 <= min_cosine <= 1
    assert len(results) == len(embeddings)
    cleaned_pages, cleaned_embeddings, audits = [], [], []
    for page_index, (raw, vectors) in enumerate(zip(results, embeddings)):
        boxes = np.asarray(raw['characters'], dtype=float).reshape(-1, 4)
        assert len(vectors) == len(boxes), 'Character/embedding count mismatch'
        assert len(raw['character_cluster_labels']) == len(boxes)
        areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
        # Largest first. Equal areas prefer the earlier original character index.
        retained, representatives, decisions = [], {}, []
        for inner in sorted(range(len(boxes)), key=lambda i: (-areas[i], i)):
            representative = inner
            for outer in retained:
                a, b = boxes[inner], boxes[outer]
                if not np.isfinite(a).all() or not np.isfinite(b).all() or areas[inner] <= 0 or areas[outer] <= 0:
                    continue
                contained = a[0] >= b[0] and a[1] >= b[1] and a[2] <= b[2] and a[3] <= b[3]
                iou = float(areas[inner] / areas[outer]) if contained else 0.0
                if not contained or iou < min_iou:
                    continue
                x, y = np.asarray(vectors[inner], dtype=float), np.asarray(vectors[outer], dtype=float)
                assert x.ndim == y.ndim == 1 and x.shape == y.shape
                denominator = np.linalg.norm(x) * np.linalg.norm(y)
                if not np.isfinite(x).all() or not np.isfinite(y).all() or not np.isfinite(denominator) or denominator <= 0:
                    continue
                cosine = float(np.clip(np.dot(x, y) / denominator, -1, 1))
                if cosine >= min_cosine:
                    representative = outer
                    decisions.append(dict(removed_original_index=inner, retained_original_index=outer,
                                          smaller_box_coverage=1.0, iou=iou, cosine=cosine))
                    break
            representatives[inner] = representative
            if representative == inner:
                retained.append(inner)
        # Keep the original relative order of surviving character appearances.
        kept = sorted(retained)
        new_index = {old: new for new, old in enumerate(kept)}
        old_to_new = [new_index[representatives[i]] for i in range(len(boxes))]
        page = copy.deepcopy(raw)
        for key in ['characters', 'character_cluster_labels', 'character_names']:
            if key in raw:
                assert len(raw[key]) == len(boxes), f'Misaligned {key}'
                page[key] = [raw[key][i] for i in kept]
        links, seen = [], set()
        for text_index, char_index in raw['text_character_associations']:
            assert 0 <= text_index < len(raw['texts']) and 0 <= char_index < len(boxes)
            link = (text_index, old_to_new[char_index])
            if link not in seen:
                seen.add(link)
                links.append(list(link))
        page['text_character_associations'] = links
        assert page['texts'] == raw['texts'] and page['ocr'] == raw['ocr']
        cleaned_pages.append(page)
        cleaned_embeddings.append([vectors[i] for i in kept])
        audits.append(dict(page_index=page_index, kept_original_indices=kept,
                           original_to_cleaned_index=old_to_new, removals=decisions))
    return dict(pages=cleaned_pages, embeddings=cleaned_embeddings, audit=audits)


character_cleanup = cached_stage('character_cleanup',
    dict(magi=stage_ref('magi_raw'), embeddings=stage_ref('embeddings'),
         cleanup=CHARACTER_CLEANUP_SETTINGS),
    lambda: remove_duplicate_characters(magi_raw_results, embeddings_per_page,
        CHARACTER_CLEANUP_SETTINGS['min_iou'], CHARACTER_CLEANUP_SETTINGS['min_cosine']))
character_results = character_cleanup['pages']
character_embeddings_per_page = character_cleanup['embeddings']
validate_magi(character_results)
# Apply only character fields; preserve the selected OCR already in memory.
for result, cleaned in zip(per_page_results, character_results):
    for key in ['characters', 'character_cluster_labels', 'character_names', 'text_character_associations']:
        if key in cleaned:
            result[key] = copy.deepcopy(cleaned[key])
print('Character counts before:', [len(r['characters']) for r in magi_raw_results])
print('Character counts after: ', [len(r['characters']) for r in character_results])
display(pd.DataFrame([dict(page_index=page['page_index'], **decision)
                      for page in character_cleanup['audit'] for decision in page['removals']]))


CACHE character_cleanup
Character counts before: [7, 7, 8]
Character counts after:  [7, 7, 8]


""


## 12. Cross-page entity matching functions

Match page-local clusters to sequence entities using the strongest member cosine similarity. Each entity accepts at most one local cluster per page.

In [35]:
def page_clusters(result, page_index):
    groups = {}
    for char_index, label in enumerate(result['character_cluster_labels']):
        cluster_id = f'PG{page_index+1}:CL{int(label)+1}' if int(label) >= 0 else f'PG{page_index+1}:C{char_index+1}:unassigned'
        groups.setdefault(cluster_id, []).append(char_index)
    return groups


def clusterize(results, embeddings, threshold=0.50):
    assert -1 <= threshold <= 1, 'Cosine threshold must be between -1 and 1'
    vectors = [np.asarray(page, dtype=float) for page in embeddings]
    assert len(results) == len(vectors)
    for result, page_vectors in zip(results, vectors):
        assert len(result['characters']) == len(page_vectors), 'Embedding count mismatch'
    entities, decisions, char_to_entity = {}, [], {}
    for page_index, result in enumerate(results):
        clusters = page_clusters(result, page_index)
        candidates = []
        for cluster_id, indices in clusters.items():
            for entity_id, entity in entities.items():
                best_score, best_pair = -float('inf'), None
                for current in indices:
                    for member in entity['members']:
                        score = float(np.dot(vectors[page_index][current], vectors[member['page_index']][member['character_index']]))
                        if score > best_score:
                            best_score = score
                            best_pair = dict(current=f'PG{page_index+1}:C{current+1}', existing=member['character_id'])
                if best_score >= threshold:
                    candidates.append(dict(cluster_id=cluster_id, entity_id=entity_id, score=best_score, best_pair=best_pair))
        candidates.sort(key=lambda candidate: candidate['score'], reverse=True)
        accepted, used_entities = {}, set()
        for candidate in candidates:
            if candidate['cluster_id'] not in accepted and candidate['entity_id'] not in used_entities:
                accepted[candidate['cluster_id']] = candidate
                used_entities.add(candidate['entity_id'])
        for cluster_id, indices in clusters.items():
            match = accepted.get(cluster_id)
            entity_id = match['entity_id'] if match else f'E{len(entities)+1}'
            if entity_id not in entities:
                entities[entity_id] = dict(clusters=[], members=[])
            entities[entity_id]['clusters'].append(cluster_id)
            for char_index in indices:
                character_id = f'PG{page_index+1}:C{char_index+1}'
                entities[entity_id]['members'].append(dict(page_index=page_index, character_index=char_index,
                                                         character_id=character_id, cluster_id=cluster_id))
                char_to_entity[character_id] = entity_id
            decisions.append(dict(page_index=page_index, cluster_id=cluster_id, entity_id=entity_id,
                                  reason='matched' if match else 'new_entity',
                                  score=match['score'] if match else None, best_pair=match['best_pair'] if match else None))
    return dict(entities=entities, char_to_entity=char_to_entity, decisions=decisions)


## 13. Reuse or create entity identities

Use the existing validated clustering threshold and save the matching audit. Character identities are scoped to this sequence.

In [36]:
entity_results = cached_stage('entities',
    dict(characters=stage_ref('character_cleanup'), threshold=ENTITY_THRESHOLD,
         algorithm='sequential_best_member_one_cluster_per_entity_per_page'),
    lambda: clusterize(character_results, character_embeddings_per_page, ENTITY_THRESHOLD))
identified_char = entity_results['entities']
char_to_entity = entity_results['char_to_entity']
assert len(char_to_entity) == sum(len(r['characters']) for r in character_results)
print('Entities:', len(identified_char))
display(pd.DataFrame(entity_results['decisions']))


CACHE entities
Entities: 9


,page_index,cluster_id,entity_id,reason,score,best_pair
0,0,PG1:CL1,E1,new_entity,NaN,None
1,0,PG1:CL2,E2,new_entity,NaN,None
2,0,PG1:CL3,E3,new_entity,NaN,None
3,0,PG1:CL4,E4,new_entity,NaN,None
4,1,PG2:CL1,E5,new_entity,NaN,None
5,1,PG2:CL2,E3,matched,0.948633,"{'current': 'PG2:C4', 'existing': 'PG1:C6'}"
6,1,PG2:CL3,E6,new_entity,NaN,None
7,1,PG2:CL4,E7,new_entity,NaN,None
8,2,PG3:CL1,E8,new_entity,NaN,None
9,2,PG3:CL2,E2,matched,0.860615,"{'current': 'PG3:C3', 'existing': 'PG1:C2'}"


## 14. Text-to-entity mapping functions

Follow cleaned text → character → entity links. Missing links remain unassociated, and multiple entities remain ambiguous.

In [37]:
def map_text_entities(results, selections, entity_map, sequence_id, page_names):
    assert len(results) == len(selections) == len(page_names)
    records = []
    for page_index, (result, decisions) in enumerate(zip(results, selections)):
        assert len(result['texts']) == len(result['ocr']) == len(decisions)
        links_by_text = {}
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']) and 0 <= char_index < len(result['characters'])
            links_by_text.setdefault(text_index, []).append(char_index)
        for text_index, (box, text, decision) in enumerate(zip(result['texts'], result['ocr'], decisions)):
            assert decision['page_index'] == page_index and decision['text_index'] == text_index and decision['box'] == box
            assert decision['selected_text'] == text
            links = []
            for char_index in dict.fromkeys(links_by_text.get(text_index, [])):
                char_id = f'PG{page_index+1}:C{char_index+1}'
                links.append(dict(character_index=char_index, character_id=char_id, entity_id=entity_map[char_id]))
            entity_ids = list(dict.fromkeys(link['entity_id'] for link in links))
            records.append(dict(sequence_id=sequence_id, page_index=page_index, page_file=page_names[page_index],
                                text_index=text_index, text_id=f'T{text_index+1}', box=box, text=text,
                                entity_id=entity_ids[0] if len(entity_ids) == 1 else None,
                                entity_ids=entity_ids, character_links=links,
                                association_status='unassociated' if not links else 'linked' if len(entity_ids) == 1 else 'ambiguous',
                                ocr_source=decision['selected_source'], ocr_reason=decision['reason'],
                                needs_review=decision['needs_review'] or len(entity_ids) > 1,
                                disagreement=decision['disagreement']))
    return records


## 15. Original text-to-entity candidates
Map every original text box to cleaned character entities without changing original links. The final artifact/punctuation flags are applied before grouping and final association.

In [38]:
text_entities = cached_stage('text_entities',
    dict(magi=stage_ref('magi_raw'), characters=stage_ref('character_cleanup'),
         selector=stage_ref('ocr_selection'), artifacts=stage_ref('artifact_labels'),entities=stage_ref('entities')),
    lambda: map_text_entities(per_page_results, ocr_selection, char_to_entity, SEQUENCE_ID,
                             [path.name for path in PAGE_FILES]))
assert len(text_entities) == sum(len(r['texts']) for r in magi_raw_results)
columns = ['page_index', 'text_id', 'text', 'entity_id', 'association_status',
           'ocr_source', 'needs_review', 'disagreement']
display(pd.DataFrame(text_entities, columns=columns))

for record in text_entities:
    record['box_id']=f"PG{PAGE_LABELS[record['page_index']]}:T{record['text_index']+1}"
    record.update(ocr_agreement[record['page_index']][record['text_index']])
    decision=ocr_selection[record['page_index']][record['text_index']]
    record.update(magi_text=decision['magi_text'],paddle_text=decision['paddle_text'])
# Keep these candidates immutable; final group speakers are resolved later.
raw_text_entities=copy.deepcopy(text_entities)


CACHE text_entities


,page_index,text_id,text,entity_id,association_status,ocr_source,needs_review,disagreement
0,0,T1,MANGA STREAM DO NOT MIRROR,E3,linked,paddle,False,True
1,0,T2,GOT YOUR TOMATO DEVIL RIGHT HERE.,E3,linked,paddle,False,False
2,0,T3,"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",E3,linked,paddle,False,True
3,0,T4,"GOOD WORK, DENJI.",E1,linked,paddle,False,False
4,0,T5,THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON T...,E1,linked,paddle,False,False
5,0,T6,THE REWARD'S 400 GRAND.,E1,linked,paddle,False,False
6,0,T7,,E1,linked,NaN,False,True
7,0,T8,THANKS !!,E3,linked,paddle,False,False
8,0,T9,I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,NaN,unassociated,paddle,False,True
9,0,T10,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,NaN,unassociated,paddle,False,True


## 16. Local YOLO bubble detection
Use only this notebook’s own stage cache, otherwise run local YOLO on each input page. No experiment cache imports.

In [39]:
BUBBLE_SHA256=file_hash(BUBBLE_MODEL)


def predict_bubbles():
    # No external cache discovery: cached_stage reads only this notebook's own cache.
    from ultralytics import YOLO
    # Weights: https://huggingface.co/PSImera/manga_bubbles_detect
    detector=YOLO(str(BUBBLE_MODEL))
    pages=[]
    try:
        for image in crop_pages:
            prediction=detector.predict(source=Image.fromarray(image),imgsz=DETECTION['imgsz'],
                conf=DETECTION['confidence'],device=0 if DEVICE=='cuda' else 'cpu',verbose=False)[0]
            height,width=image.shape[:2];candidates=[]
            for box,score in zip(prediction.boxes.xyxy.cpu().tolist(),prediction.boxes.conf.cpu().tolist()):
                a,b,c,d=box
                bounds=[max(0,math.floor(a)),max(0,math.floor(b)),min(width,math.ceil(c)),min(height,math.ceil(d))]
                if bounds[2]>bounds[0] and bounds[3]>bounds[1]:candidates.append(dict(box=bounds,confidence=float(score)))
            candidates.sort(key=lambda b:(b['box'][1],-b['box'][0]))
            pages.append(dict(candidates=candidates,source='own_local_yolo'))
            del prediction
    finally:
        del detector;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    return pages


bubble_pages=cached_stage('contour_bubbles',dict(weights_sha256=BUBBLE_SHA256,settings=DETECTION,
    ultralytics=version('ultralytics'),image_mode='original_RGB',algorithm=function_digest(predict_bubbles)),predict_bubbles)
print('Bubble candidates:',[len(p['candidates']) for p in bubble_pages])


CACHE contour_bubbles
Bubble candidates: [11, 8, 7]


## 17. Stage 1 punctuation gate, then Stage 2 contour grouping/roles
Remove Gemma Artifacts. Empty-Paddle + punctuation-only Magi requires a qualifying bubble/background mask in a 20% padded YOLO crop. All other retained texts proceed unchanged. Run the existing outline/faint/overlap grouping and contour roles on survivors only. Any explicit Magi tail link on a member forces its group SPOKEN. No Artifact rescue.

In [40]:
CONTOUR_INFO=dict(module_sha256=file_hash(CONTOUR_MODULE_PATH),
    grouping_sha256=file_hash(CONTOUR_GROUPING_PATH),check=CONTOUR_SETTINGS,
    region=REGION_SETTINGS,faint_region=FAINT_REGION_SETTINGS,group_faint=GROUP_FAINT_SETTINGS)


def filter_artifact_punctuation():
    filtered=copy.deepcopy(artifact_initial)
    for page_index,(selections,labels) in enumerate(zip(ocr_selection,filtered)):
        text_boxes=[d['box'] for d in selections]
        image=Image.fromarray(crop_pages[page_index])
        for index,(decision,label) in enumerate(zip(selections,labels)):
            if label['delete_ocr'] or decision['paddle_text'].strip() or not punctuation_only(decision['magi_text']):continue
            found=False;checked_candidate=None
            a,b,c,d=decision['box'];area=max(1,(c-a)*(d-b))
            for candidate_id,candidate in enumerate(bubble_pages[page_index]['candidates']):
                if contour_model.contours.overlap(decision['box'],candidate['box'])/area<.8:continue
                result=contour_model.find_background_regions(candidate,image,text_boxes,index)
                region=result['best']
                if (region is not None and not region['leak_suspected']
                    and region['visible_points']>=REGION_SETTINGS['min_visible_points']):
                    found=True;checked_candidate=candidate_id;break
            label.update(punctuation_bubble_checked=True,punctuation_bubble_found=found,
                punctuation_candidate_id=checked_candidate)
            if not found:
                label.update(artifact_label='Artifact',delete_ocr=True,
                    deletion_reason='Empty Paddle + Magi punctuation has no qualifying bubble mask')
    return filtered


ocr_agreement=cached_stage('artifact_filter',dict(labels=stage_ref('artifact_labels'),
    bubbles=stage_ref('contour_bubbles'),contour=CONTOUR_INFO,coverage=.8,
    algorithm=function_digest(filter_artifact_punctuation)),filter_artifact_punctuation)
# The same final deletion flags feed grouping, association routing and all exports.
for record in text_entities:
    record.update(ocr_agreement[record['page_index']][record['text_index']])
raw_text_entities=copy.deepcopy(text_entities)


def contour_texts(page_index):
    texts=[];page_number=PAGE_LABELS[page_index]
    for index,decision in enumerate(ocr_selection[page_index]):
        filtered=ocr_agreement[page_index][index]
        if filtered['delete_ocr']:continue
        texts.append(dict(box_id=f'PG{page_number}:T{index+1}',text_index=index,box=decision['box'],
            text=decision['selected_text'],magi_text=decision['magi_text'],paddle_text=decision['paddle_text'],
            ocr_agreement=filtered['ocr_agreement']))
    return texts


def make_contour_roles():
    pages=[]
    for page_index,path in enumerate(PAGE_FILES):
        analysis,_=contour_model.analyze_page(bubble_pages[page_index]['candidates'],
            Image.fromarray(crop_pages[page_index]),contour_texts(page_index),PAGE_LABELS[page_index])
        for row in analysis['rows']:
            row.update(sequence_id=SEQUENCE_ID,page_index=page_index,page_file=path.name)
            row['contour_text_role']=row['text_role'];row['contour_original_reason']=row['reason']
            member_ids=set(row['box_ids']);tail_ids=[];tail_members=[]
            raw=magi_raw_results[page_index]
            for text_index,tail_index in raw['text_tail_associations']:
                assert 0<=text_index<len(raw['texts']) and 0<=tail_index<len(raw['tails'])
                bid=f'PG{PAGE_LABELS[page_index]}:T{text_index+1}'
                if bid in member_ids:tail_ids.append(tail_index);tail_members.append(bid)
            row['tail_ids']=sorted(set(tail_ids));row['tail_member_box_ids']=sorted(set(tail_members))
            row['tail_override']=bool(tail_ids)
            if row['tail_override']:
                row['text_role']='SPOKEN';row['reason']='Magi text-to-tail association forces SPOKEN'
        pages.append(analysis)
    return pages


contour_pages=cached_stage('contour_roles',
    dict(magi=stage_ref('magi_raw'),selector=stage_ref('ocr_selection'),filter=stage_ref('artifact_filter'),
         bubbles=stage_ref('contour_bubbles'),contour=CONTOUR_INFO,
         algorithm=function_digest(make_contour_roles),adapter=function_digest(contour_texts)),make_contour_roles)
contour_rows=[r for page in contour_pages for r in page['rows']]
expected={r['box_id'] for r in raw_text_entities if not r['delete_ocr']}
emitted=[bid for row in contour_rows for bid in row['box_ids']]
assert len(emitted)==len(set(emitted)) and set(emitted)==expected,'Missing/duplicated surviving text'
display(pd.DataFrame(contour_rows).reindex(columns=['box_id','box_ids','text','text_role','reason','needs_review']))


CACHE artifact_filter
CACHE contour_roles


,box_id,box_ids,text,text_role,reason,needs_review
0,PG1:T2,[PG1:T2],GOT YOUR TOMATO DEVIL RIGHT HERE.,SPOKEN,Magi text-to-tail association forces SPOKEN,False
1,PG1:T3,[PG1:T3],"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",SPOKEN,Magi text-to-tail association forces SPOKEN,False
2,PG1:M1,"[PG1:T4, PG1:T5]","GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A...",SPOKEN,Magi text-to-tail association forces SPOKEN,False
3,PG1:T6,[PG1:T6],THE REWARD'S 400 GRAND.,SPOKEN,Magi text-to-tail association forces SPOKEN,False
4,PG1:T8,[PG1:T8],THANKS !!,SPOKEN,Magi text-to-tail association forces SPOKEN,False
5,PG1:T9,[PG1:T9],I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,SPOKEN,Magi text-to-tail association forces SPOKEN,False
6,PG1:T10,[PG1:T10],AND THEN THERE'S THE COMMISSION FOR REFERRING ...,SPOKEN,Magi text-to-tail association forces SPOKEN,False
7,PG1:T11,[PG1:T11],ONLY 70 GRAND LEFT...,SPOKEN,Magi text-to-tail association forces SPOKEN,False
8,PG2:T1,[PG2:T1],AFTER PAYING THE WATER BILL,SPOKEN,Magi text-to-tail association forces SPOKEN,False
9,PG2:T2,[PG2:T2],"AND THE OTHER LOANS,",SPOKEN,Magi text-to-tail association forces SPOKEN,False


## 18. Route group labels to narrator or sequence character
Narrated groups use NARRATION. Spoken groups use the single distinct entity among their original Magi text-character links. A partial link is used with review; missing or conflicting entities remain unresolved. We do not guess a new association from the combined rectangle. Preserve all member links and OCR/contour review reasons.


In [41]:
def resolve_group_speaker(role,members):
    entity_ids=list(dict.fromkeys(entity for member in members for entity in member['entity_ids']))
    missing=[m['box_id'] for m in members if not m['entity_ids']]
    links=[dict(box_id=m['box_id'],entity_ids=m['entity_ids'],character_links=m['character_links'],
                association_status=m['association_status']) for m in members]
    if role=='NARRATED':speaker,status,review='NARRATION','narrator',False
    elif len(entity_ids)>1:speaker,status,review=None,'conflicting_entities',True
    elif not entity_ids:speaker,status,review=None,'unassociated',True
    elif missing:speaker,status,review=entity_ids[0],'partially_linked',True
    else:speaker,status,review=entity_ids[0],'linked',False
    return dict(speaker=speaker,speaker_status=status,speaker_needs_review=review,
                candidate_entity_ids=entity_ids,unlinked_member_ids=missing,member_associations=links)


def make_group_speakers():
    lookup={r['box_id']:r for r in raw_text_entities if not r['delete_ocr']}
    rows=[]
    for contour in contour_rows:
        members=[lookup[bid] for bid in contour['box_ids']]
        assert all(m['page_index']==contour['page_index'] for m in members)
        row=copy.deepcopy(contour)
        row.update(resolve_group_speaker(row['text_role'],members))
        ocr_review=any(ocr_selection[m['page_index']][m['text_index']]['needs_review'] for m in members)
        row['needs_review']=bool(row['needs_review'] or row['speaker_needs_review'] or ocr_review)
        row['ocr_needs_review']=ocr_review
        rows.append(row)
    return rows


grouped_records=cached_stage('contour_speakers',
    dict(roles=stage_ref('contour_roles'),associations=stage_ref('text_entities'),filter=stage_ref('artifact_filter'),
         algorithm=function_digest(make_group_speakers),routing=function_digest(resolve_group_speaker)),make_group_speakers)
# Export an audit of all original boxes, including deleted boxes, without reordering them.
by_member={bid:row for row in grouped_records for bid in row['box_ids']}
box_records=[]
for original in raw_text_entities:
    row=copy.deepcopy(original);group=by_member.get(row['box_id'])
    if row['delete_ocr']:
        assert group is None
        row.update(group_id=None,text_role=None,speaker=None,speaker_status='deleted')
    else:
        assert group is not None
        row.update(group_id=group['box_id'],text_role=group['text_role'],speaker=group['speaker'],
            speaker_status=group['speaker_status'],needs_review=group['needs_review'],contour_reason=group['reason'],
            tail_override=group['tail_override'],tail_ids=group['tail_ids'],
            contour_text_role=group['contour_text_role'])
    box_records.append(row)
display(pd.DataFrame(grouped_records).reindex(columns=['box_id','box_ids','speaker','speaker_status','text_role','text','needs_review']))


CACHE contour_speakers


,box_id,box_ids,speaker,speaker_status,text_role,text,needs_review
0,PG1:T2,[PG1:T2],E3,linked,SPOKEN,GOT YOUR TOMATO DEVIL RIGHT HERE.,False
1,PG1:T3,[PG1:T3],E3,linked,SPOKEN,"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",False
2,PG1:M1,"[PG1:T4, PG1:T5]",E1,linked,SPOKEN,"GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A...",False
3,PG1:T6,[PG1:T6],E1,linked,SPOKEN,THE REWARD'S 400 GRAND.,False
4,PG1:T8,[PG1:T8],E3,linked,SPOKEN,THANKS !!,False
5,PG1:T9,[PG1:T9],NaN,unassociated,SPOKEN,I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,True
6,PG1:T10,[PG1:T10],NaN,unassociated,SPOKEN,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,True
7,PG1:T11,[PG1:T11],E3,linked,SPOKEN,ONLY 70 GRAND LEFT...,False
8,PG2:T1,[PG2:T1],E3,linked,SPOKEN,AFTER PAYING THE WATER BILL,False
9,PG2:T2,[PG2:T2],E3,linked,SPOKEN,"AND THE OTHER LOANS,",False


## 19. Before/after single-page viewers
Select one page at a time. Before shows Gemma labels; after shows final groups, SPOKEN/NARRATED and entities. TAIL marks the hard override. Deleted boxes are excluded by default; show_page(0,show_deleted=True) includes them. Figures are closed after display.

In [42]:
def draw_box(ax,box,color,label=None,style='-'):
    a,b,c,d=box
    ax.add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,edgecolor=color,linewidth=2,linestyle=style))
    if label:ax.text(a,b,label,fontsize=8,color='black',bbox=dict(facecolor=color,alpha=.8))


def center(box):
    a,b,c,d=box;return ((a+c)/2,(b+d)/2)


def show_page(page_index=0,groups=True,member_boxes=True,characters=True,associations=False,
              panels=False,show_deleted=False,show_ocr=True,save_path=None):
    result=per_page_results[page_index]
    fig,ax=plt.subplots(figsize=(13,20));ax.imshow(crop_pages[page_index])
    if panels:
        for box in result.get('panels',[]):draw_box(ax,box,'lime')
    cmap=plt.get_cmap('tab20',max(1,len(identified_char)))
    colors={entity:cmap(i) for i,entity in enumerate(identified_char)}
    if characters:
        for i,box in enumerate(result['characters']):
            entity=char_to_entity[f'PG{page_index+1}:C{i+1}']
            draw_box(ax,box,colors[entity],f'C{i+1} / {entity}')
    if groups:
        for row in grouped_records:
            if row['page_index']!=page_index:continue
            color='yellow' if row['text_role']=='SPOKEN' else 'grey'
            if row['visible_segments']:
                for segment in row['visible_segments']:
                    points=np.asarray(segment);ax.plot(points[:,0],points[:,1],color=color,linewidth=3)
            else:
                for box in row['member_text_boxes']:draw_box(ax,box,color)
            if member_boxes:
                for bid,box in zip(row['box_ids'],row['member_text_boxes']):draw_box(ax,box,color,bid,style='--')
            label=row['box_id']+' '+row['text_role']+' / '+(row['speaker'] or row['speaker_status'])
            if row.get('tail_override'):label+=' [TAIL]'
            if row['needs_review']:label+=' [review]'
            if show_ocr:label+=' | '+row['text'][:90]
            a,b,_,_=row['text_box'];ax.text(a,b,label,fontsize=8,bbox=dict(facecolor=color,alpha=.8))
    if associations:
        for record in raw_text_entities:
            if record['page_index']!=page_index or record['delete_ocr']:continue
            x,y=center(record['box'])
            for link in record['character_links']:
                u,v=center(result['characters'][link['character_index']])
                ax.plot([x,u],[y,v],color=colors[link['entity_id']],linewidth=1,alpha=.8)
    if show_deleted:
        for row in box_records:
            if row['page_index']==page_index and row['delete_ocr']:
                draw_box(ax,row['box'],'red','DELETED '+row['box_id'],style='--')
    ax.set_title(f'{PAGE_FILES[page_index].name} | yellow: spoken | grey: narrated | red: deleted')
    ax.axis('off');fig.tight_layout()
    if save_path:fig.savefig(save_path,bbox_inches='tight',dpi=150)
    buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()));return fig


def show_artifact_page(page_index=0):
    fig,ax=plt.subplots(figsize=(11,16));ax.imshow(crop_pages[page_index])
    for index,(decision,label) in enumerate(zip(ocr_selection[page_index],artifact_initial[page_index])):
        color={'Text':'limegreen','Artifact':'red','Review':'darkorange'}[label['gemma_label']]
        draw_box(ax,decision['box'],color,f"PG{PAGE_LABELS[page_index]}:T{index+1} {label['gemma_label']}")
    ax.set_title('Before contour | Gemma labels | '+PAGE_FILES[page_index].name)
    ax.axis('off');buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))


def make_page_viewer(title,render,key):
    if widgets is None:
        print('ipywidgets unavailable: call show_artifact_page(0) or show_page(0)');return
    registry=globals().setdefault('_standalone_viewers',{})
    old=registry.pop(key,None)
    if old:
        old[0].unobserve_all()
        for item in old:item.close()
    picker=widgets.Dropdown(options=[(f'PG{PAGE_LABELS[i]} '+p.name,i) for i,p in enumerate(PAGE_FILES)],description='Page')
    output=widgets.Output()
    def refresh(change=None):
        with output:
            clear_output(wait=True);render(picker.value)
    picker.observe(refresh,names='value')
    panel=widgets.VBox([widgets.HTML(value='<b>'+title+'</b>'),picker,output])
    registry[key]=[picker,output,panel]
    display(panel);refresh()


make_page_viewer('Before contour: Gemma artifact decisions',show_artifact_page,'before')
make_page_viewer('After filtering: roles and character entities',show_page,'after')


## 20. Optional group inspection
Call inspect_group(group_id) manually to inspect masks, faint retries and visible contour evidence. It uses only in-memory images/detections; no model or external cache reads.

In [43]:
inspection_pages={}


def inspect_group(group_id):
    row=next(r for r in grouped_records if r['box_id']==group_id)
    page_index=row['page_index']
    if page_index not in inspection_pages:
        analysis,debug=contour_model.analyze_page(bubble_pages[page_index]['candidates'],
            Image.fromarray(crop_pages[page_index]),contour_texts(page_index),PAGE_LABELS[page_index])
        inspection_pages[page_index]=dict(analysis=analysis,debug=debug)
    cached=inspection_pages[page_index]
    index=next(i for i,r in enumerate(cached['analysis']['rows']) if r['box_id']==group_id)
    print(group_id,row['text_role'],'|',row['speaker'] or row['speaker_status'],'| Review:',row['needs_review'])
    print(row['text']);print('Members:',row['box_ids']);print('Reason:',row['reason'])
    for attempt in cached['debug'][index]['attempts']:
        checked=attempt['result'];region=checked['best'];left,top,_,_=checked['bounds']
        fig,axes=plt.subplots(1,4,figsize=(18,6))
        axes[0].imshow(checked['crop'])
        for a,b,c,d in row['member_text_boxes']:
            axes[0].add_patch(patches.Rectangle((a-left,b-top),c-a,d-b,fill=False,edgecolor='deepskyblue'))
        axes[0].set_title(f"YOLO {attempt['candidate_id']} + 20% per side")
        axes[1].imshow(region['mask'] if region else np.zeros(checked['crop'].shape[:2]),cmap='gray')
        axes[1].set_title('White / solid background region')
        axes[2].imshow(checked.get('retry_barrier',checked.get('barrier',np.zeros(checked['crop'].shape[:2]))),cmap='gray')
        axes[2].set_title('Faint-edge barriers (retry only)')
        overlay=checked['crop'].copy()
        if region:
            points=region['points']
            for i in range(len(points)):
                nxt=(i+1)%len(points)
                if not (region['visible'][i] and region['visible'][nxt]):continue
                color=(255,220,0) if region['curved'][i] else (0,200,255) if region['flat'][i] else (255,140,0)
                cv2.line(overlay,tuple(points[i]),tuple(points[nxt]),color,2)
            print('Region:',region['text_role'],region['reason'],'| Pass:',region['extraction_pass'])
        axes[3].imshow(overlay);axes[3].set_title('Curves yellow | flat cyan | sharp orange')
        print('Retry:',checked['retry_attempted'],'| First:',checked['first_pass'])
        if checked.get('retry_reason'):print('Retry result:',checked['retry_reason'])
        for ax in axes:ax.axis('off')
        fig.tight_layout();plt.show()
    display(pd.DataFrame(row['member_associations']))
    display(pd.DataFrame([{k:v for k,v in row.items() if k not in ['contour_points','visible_segments','member_associations']}]))


# Optional manual call: inspect_group(grouped_records[0]['box_id'])


## 21. Transcript and audit exports
Same labels.jsonl entry schema: speaker/text only. Character names are sequence-local; unresolved spoken text uses UNKNOWN. Save deleted original boxes, artifact decisions, grouping, roles, tails, entities and resource manifest.

In [44]:
def build_transcript(sequence_id,records,page_files,sequence_files,entity_ids):
    # Stable sequence-local names; these are predicted identities, not GT character matches.
    ordered_entities=sorted(entity_ids,key=lambda eid:(int(eid[1:]) if eid.startswith('E') and eid[1:].isdigit() else float('inf'),eid))
    names={entity:f'character{i+1}' for i,entity in enumerate(ordered_entities)}
    page_positions={str(path.resolve()):i for i,path in enumerate(sequence_files)}
    pages=[[] for _ in sequence_files]
    for row in sorted(records,key=lambda r:(r['page_index'],r['text_index'])):
        page=page_positions[str(page_files[row['page_index']].resolve())]
        speaker='NARRATION' if row['text_role']=='NARRATED' else names.get(row['speaker'],'UNKNOWN')
        pages[page].append(dict(speaker=speaker,text=row['text']))
    return dict(sequence_id=sequence_id,pages=pages),names


# Keep detailed exports; the transcript below has exactly the labels.jsonl contract.
# sequence_files is the selected explicit image list or original dataset page list.
transcript,entity_to_character=build_transcript(SEQUENCE_ID,grouped_records,PAGE_FILES,sequence_files,identified_char)
write_json(OUTPUT_DIR/'transcript.json',transcript)
# One sequence object per physical line. Replace this run's file rather than appending duplicates.
(OUTPUT_DIR/'transcript.jsonl').write_text(json.dumps(transcript,ensure_ascii=False)+'\n',encoding='utf-8')
write_json(OUTPUT_DIR/'transcript_speaker_mapping.json',entity_to_character)


dialogue_pages=[[] for _ in PAGE_FILES]
for row in grouped_records:
    dialogue_pages[row['page_index']].append(dict(group_id=row['box_id'],box_ids=row['box_ids'],
        speaker=row['speaker'],speaker_status=row['speaker_status'],text_role=row['text_role'],
        text=row['text'],needs_review=row['needs_review']))
write_json(OUTPUT_DIR/'dialogue_pages.json',dialogue_pages)
write_json(OUTPUT_DIR/'group_predictions.json',grouped_records)
pd.DataFrame(grouped_records).to_csv(OUTPUT_DIR/'group_predictions.csv',index=False)
write_json(OUTPUT_DIR/'box_predictions.json',box_records)
pd.DataFrame(box_records).to_csv(OUTPUT_DIR/'box_predictions.csv',index=False)
deleted=[r for r in box_records if r['delete_ocr']]
write_json(OUTPUT_DIR/'deleted_ocr.json',deleted)
pd.DataFrame(deleted).to_csv(OUTPUT_DIR/'deleted_ocr.csv',index=False)
write_json(OUTPUT_DIR/'grouping_audit.json',[p['grouping'] for p in contour_pages])
write_json(OUTPUT_DIR/'entities.json',entity_results)
write_json(OUTPUT_DIR/'run_metadata.json',dict(sequence_id=SEQUENCE_ID,pages=PAGE_METADATA,
    narration_method='contour_narrator2',contour=CONTOUR_INFO,bubble_weights_sha256=BUBBLE_SHA256,
    bubble_settings=DETECTION,character_checkpoint=CHECKPOINT_INFO,entity_threshold=ENTITY_THRESHOLD,
    association_head=str(ASSOCIATION_HEAD_PATH) if ASSOCIATION_HEAD_PATH else 'original_magi',
    tail_override=True,ocr_filter=dict(method='Paddle-only local Gemma',
        agreement_deletion=False,punctuation_fallback_requires_bubble=True),dependencies=RESOURCE_MANIFEST,
    manifest=str(MANIFEST_PATH),stages=manifest['stages']))
lines=[]
for path,page in zip(sequence_files,transcript['pages']):
    lines.append(f'[Page {path.name}]')
    lines.extend(f"{row['speaker']}: {row['text']}" for row in page)
    lines.append('')
(OUTPUT_DIR/'transcript.txt').write_text('\n'.join(lines),encoding='utf-8')

display(pd.DataFrame([dict(page=path.name,detected=len(magi_raw_results[i]['texts']),
    deleted=sum(r['delete_ocr'] for r in box_records if r['page_index']==i),
    output_rows=len(dialogue_pages[i]),spoken=sum(r['text_role']=='SPOKEN' for r in dialogue_pages[i]),
    narrated=sum(r['text_role']=='NARRATED' for r in dialogue_pages[i]),
    review=sum(r['needs_review'] for r in dialogue_pages[i])) for i,path in enumerate(PAGE_FILES)]))
print('Saved:',OUTPUT_DIR)
# Optional annotated page export (no model work):
# for i,path in enumerate(PAGE_FILES):
#     figure=show_page(i,save_path=OUTPUT_DIR/f'annotated_{path.stem}.png')
#     plt.close(figure)

write_json(OUTPUT_DIR/'artifact_decisions.json',artifact_initial)
write_json(OUTPUT_DIR/'artifact_filter_audit.json',ocr_agreement)
if SAVE_ANNOTATED_PAGES:
    for i,path in enumerate(PAGE_FILES):
        show_page(i,save_path=OUTPUT_DIR/f'annotated_PG{PAGE_LABELS[i]}.png')


,page,detected,deleted,output_rows,spoken,narrated,review
0,01.png,11,2,8,8,0,2
1,02.png,9,1,7,7,0,0
2,03.png,8,1,6,6,0,0


Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_artifact_contour_full_inference\seq_2fae297a18af2a57\a922ef5f36d9a3aa\outputs


## 22. Optional final transcript OCR correction
Separate task from artifact classification. Uses local Gemma with the existing correction prompt. Preserves raw transcript, entry count and speakers. Set CLEAN_TRANSCRIPT=False to export original selected OCR; punctuation-only text stays unchanged.

In [45]:
# This final stage changes transcript text only, after grouping and speaker assignment.
GEMMA_EXAMPLES=[
    {'role': 'system', 'content':
     'You restore English manga dialogue corrupted by OCR. '
     'Repair obvious broken words. Preserve legitimate names, valid single-letter words such as I and A, '
     'Preserve Punctutaion and Punctuation Only Sentences'
     'Remove nonsensical inserted letter sequences and accidental isolated letters that interrupt a sentence. '
     'capitalization, meaning, punctuation and unfinished sentences. '
     'Do not rewrite the dialogue, complete it or answer it. '
     'Preserve numbers, amounts, times and units; never delete a number while repairing nearby words. '
     'Repair obvious O/0 OCR confusion inside a numeric amount, such as 17O GRAND to 170 GRAND. '
     'any single character repeated multiple times could be punctuations mis interpreted '
     'Return only the restored text, with no explanation or quotation marks.'},
    {'role': 'user', 'content': 'PLEASE Q OPEN THE WINDOW.'},
    {'role': 'assistant', 'content': 'PLEASE OPEN THE WINDOW.'},
    {'role': 'user', 'content': 'I CAN HEAR XZQWZXQ SOMEONE OUTSIDE...'},
    {'role': 'assistant', 'content': 'I CAN HEAR SOMEONE OUTSIDE...'},
    {'role': 'user', 'content': 'AH... SHIROGANE, WAIT!'},
    {'role': 'assistant', 'content': 'AH... SHIROGANE, WAIT!'},
    {'role': 'user', 'content': '.....'},
    {'role': 'assistant', 'content': '.....'},
    {'role': 'user', 'content': 'PLEASE S OPEN THE WINDOW.'},
    {'role': 'assistant', 'content': 'PLEASE OPEN THE WINDOW.'},
    # Repair OCR words and numeric confusions without losing amounts or times.
    {'role': 'user', 'content': "I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AND INTEREST PAYMENTS."},
    {'role': 'assistant', 'content': "I'M TAKING OUT 170 GRAND TO COVER YOUR LOAN AND INTEREST PAYMENTS."},
    {'role': 'user', 'content': 'im worlkng in 20 minutes'},
    {'role': 'assistant', 'content': 'im working in 20 minutes'},
    {'role': 'user', 'content': 'WE HAVE 3 MINUTFS LEFT!'},
    {'role': 'assistant', 'content': 'WE HAVE 3 MINUTES LEFT!'},
    {'role': 'user', 'content': 'HE PA1D 500 YEN FOR IT.'},
    {'role': 'assistant', 'content': 'HE PAID 500 YEN FOR IT.'},
    {'role': 'user', 'content': 'I OWE YOU 170 GRAND.'},
    {'role': 'assistant', 'content': 'I OWE YOU 170 GRAND.'},
]
FORCE.setdefault('gemma_transcript',False)
STAGE_VERSIONS['gemma_transcript']=1
# Always rebuild from original group OCR, so rerunning cleanup cannot repeatedly edit its own output.
raw_transcript,_=build_transcript(SEQUENCE_ID,grouped_records,PAGE_FILES,sequence_files,identified_char)
# Keep the exact original OCR transcript before changing any final transcript exports.
write_json(OUTPUT_DIR/'transcript_raw.json',raw_transcript)
(OUTPUT_DIR/'transcript_raw.jsonl').write_text(json.dumps(raw_transcript,ensure_ascii=False)+'\n',encoding='utf-8')
raw_lines=[]
for path,page in zip(sequence_files,raw_transcript['pages']):
    raw_lines.append(f'[Page {path.name}]')
    raw_lines.extend(f"{row['speaker']}: {row['text']}" for row in page)
    raw_lines.append('')
(OUTPUT_DIR/'transcript_raw.txt').write_text('\n'.join(raw_lines),encoding='utf-8')
assert (GEMMA_MODEL_PATH/'config.json').is_file(),f'Missing local Gemma: {GEMMA_MODEL_PATH}'
GEMMA_IDENTITY=dict(path=str(GEMMA_MODEL_PATH),
    files={path.name:dict(size=path.stat().st_size,mtime_ns=path.stat().st_mtime_ns,
                         sha256=file_hash(path) if path.suffix=='.json' else None)
           for path in sorted(GEMMA_MODEL_PATH.iterdir())
           if path.is_file() and path.suffix in {'.json','.safetensors','.bin','.model'}})


def correct_transcript_text(text,model,tokenizer):
    if not text.strip() or punctuation_only(text):return text
    messages=GEMMA_EXAMPLES+[{'role':'user','content':text}]
    inputs=tokenizer.apply_chat_template(messages,add_generation_prompt=True,
        tokenize=True,return_dict=True,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output=model.generate(**inputs,do_sample=False,
            max_new_tokens=max(256,inputs['input_ids'].shape[1]*2))
    return tokenizer.decode(output[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()


def make_gemma_transcript():
    cleaned=copy.deepcopy(raw_transcript);audit=[]
    if not CLEAN_TRANSCRIPT or not any(row['text'].strip() and not punctuation_only(row['text']) for page in cleaned['pages'] for row in page):
        return dict(transcript=cleaned,audit=audit)
    from transformers import AutoTokenizer,Gemma3ForConditionalGeneration,BitsAndBytesConfig
    assert torch.cuda.is_available(),'Gemma cleanup requires your CUDA godfather.venv kernel.'
    compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    # Model/tokenizer: https://huggingface.co/google/gemma-3-4b-it
    tokenizer=AutoTokenizer.from_pretrained(str(GEMMA_MODEL_PATH),local_files_only=True)
    model=Gemma3ForConditionalGeneration.from_pretrained(str(GEMMA_MODEL_PATH),local_files_only=True,
        device_map='auto',dtype=compute_dtype,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                                             bnb_4bit_compute_dtype=compute_dtype)).eval()
    try:
        # One transcript entry at a time; never send speaker IDs to the correction model.
        seen={}
        for page_index,page in enumerate(cleaned['pages']):
            for row_index,row in enumerate(page):
                original=row['text']
                if original not in seen:seen[original]=correct_transcript_text(original,model,tokenizer)
                answer=seen[original]
                row['text']=answer if answer else original
                audit.append(dict(page_index=page_index,row_index=row_index,speaker=row['speaker'],
                    raw_text=original,cleaned_text=row['text'],changed=row['text']!=original,
                    empty_output_fallback=bool(original.strip() and not answer)))
    finally:
        del model,tokenizer
        gc.collect()
        torch.cuda.empty_cache()
    return dict(transcript=cleaned,audit=audit)


# A compatible cache skips Gemma loading and generation on subsequent runs.
gemma_cleanup=cached_stage('gemma_transcript',dict(input_transcript=digest(raw_transcript),enabled=CLEAN_TRANSCRIPT,
    model=GEMMA_IDENTITY,prompt=GEMMA_EXAMPLES,quantization='nf4',generation='greedy_dynamic_max_tokens_v1',
    transformers=version('transformers'),bitsandbytes=version('bitsandbytes'),
    algorithm=function_digest(make_gemma_transcript),corrector=function_digest(correct_transcript_text)),make_gemma_transcript)
cleaned_transcript=gemma_cleanup['transcript']
assert cleaned_transcript['sequence_id']==raw_transcript['sequence_id']
assert len(cleaned_transcript['pages'])==len(raw_transcript['pages'])
for original_page,cleaned_page in zip(raw_transcript['pages'],cleaned_transcript['pages']):
    assert len(original_page)==len(cleaned_page)
    assert [r['speaker'] for r in original_page]==[r['speaker'] for r in cleaned_page]
    assert all(set(r)=={'speaker','text'} for r in cleaned_page)
def filter_final_sfx(source):
    # Apply after cleanup, including when the cleaned transcript came from cache.
    filtered=copy.deepcopy(source);removed=[]
    for page_index,page in enumerate(filtered['pages']):
        kept=[]
        for row_index,row in enumerate(page):
            if 'sfx' in row['text'].casefold():
                removed.append(dict(page_index=page_index,row_index=row_index,
                    speaker=row['speaker'],text=row['text'],reason='final text contains SFX'))
            else:
                kept.append(row)
        filtered['pages'][page_index]=kept
    return filtered,removed


# Final transcript files exclude entire entries containing SFX (case-insensitive).
transcript,sfx_removed=filter_final_sfx(cleaned_transcript)
write_json(OUTPUT_DIR/'transcript_sfx_filter_audit.json',sfx_removed)
print('Final SFX filter:',len(sfx_removed),'entries removed')
write_json(OUTPUT_DIR/'transcript.json',transcript)
(OUTPUT_DIR/'transcript.jsonl').write_text(json.dumps(transcript,ensure_ascii=False)+'\n',encoding='utf-8')
lines=[]
for path,page in zip(sequence_files,transcript['pages']):
    lines.append(f'[Page {path.name}]')
    lines.extend(f"{row['speaker']}: {row['text']}" for row in page)
    lines.append('')
(OUTPUT_DIR/'transcript.txt').write_text('\n'.join(lines),encoding='utf-8')
write_json(OUTPUT_DIR/'transcript_cleanup_audit.json',gemma_cleanup['audit'])
# Detailed group/box/dialogue exports retain original OCR; cleaned text is in transcript files above.
metadata=json.loads((OUTPUT_DIR/'run_metadata.json').read_text(encoding='utf-8'))
metadata['transcript_cleanup']=dict(model=GEMMA_IDENTITY,stage=stage_ref('gemma_transcript'),
    changed_entries=sum(row['changed'] for row in gemma_cleanup['audit']),raw_export='transcript_raw.jsonl')
metadata['final_sfx_filter']=dict(rule='case-insensitive substring SFX',removed_entries=len(sfx_removed))
metadata['stages']=manifest['stages']
write_json(OUTPUT_DIR/'run_metadata.json',metadata)
print('Gemma cleanup:',len(gemma_cleanup['audit']),'entries;',sum(r['changed'] for r in gemma_cleanup['audit']),'changed')


CACHE gemma_transcript
Gemma cleanup: 21 entries; 6 changed


## 23. Print final transcript
Final transcript after the optional correction stage.

In [46]:
# Print the cleaned transcript in the development/labels.jsonl structure.
print(json.dumps(transcript,ensure_ascii=False,indent=2))


{
  "sequence_id": "seq_2fae297a18af2a57",
  "pages": [
    [
      {
        "speaker": "character3",
        "text": "GOT YOUR TOMATO DEVIL RIGHT HERE."
      },
      {
        "speaker": "character3",
        "text": "IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT."
      },
      {
        "speaker": "character1",
        "text": "GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON THE BLACK MARKET."
      },
      {
        "speaker": "character1",
        "text": "THE REWARD'S 400 GRAND."
      },
      {
        "speaker": "character3",
        "text": "THANKS!!"
      },
      {
        "speaker": "UNKNOWN",
        "text": "I’M TAKING TO GRAND TO COVER YOUR LOAN AND INTEREST PAYMENTS."
      },
      {
        "speaker": "UNKNOWN",
        "text": "AND THEN THERE'S THE COMMISSION FOR REFERRING YOL AND ASSOCIATED FEES..."
      },
      {
        "speaker": "character3",
        "text": "ONLY 70 GRAND LEFT..."
      }
    ],
    [
      {
        "speaker": "chara

## 24. Full interactive final viewer
Select one page and toggle panels, text boxes, characters, tails, contour outlines, concatenated groups, OCR/source, SPOKEN/NARRATED labels, IDs, entity numbers/colors, same-entity connections and text-character/text-tail associations. Entity colors stay consistent across pages. Filter one entity or view all.

Same-entity lines connect appearances assigned to the same **final entity** on the selected page. Text-character lines show the final spoken routing by default; `raw candidate links` reveals original candidate links for diagnosis, including conflicting ones. Deleted text remains hidden unless enabled. The viewer uses in-memory results only and never runs models or reads another cache. Changing controls replaces the displayed page, without adding copies.

In [ ]:
# Viewer only: no changes to predictions, caches, character embeddings or transcripts.
from matplotlib.lines import Line2D


def final_entity_colors():
    ordered=sorted(identified_char,key=lambda eid:(int(eid[1:]) if eid.startswith('E') and eid[1:].isdigit() else 10**9,eid))
    # One distinct hue per sequence entity; stable across all selected pages.
    palette=plt.get_cmap('hsv',max(2,len(ordered)+1))
    return {entity:palette(i) for i,entity in enumerate(ordered)}


def show_final_page(page_index=0,ocr_source='selected',entity_filter='All',
    panels=False,texts=True,characters=True,tails=False,groups=False,contours=True,
    same_entity=False,text_character=True,text_tail=False,ocr=True,roles=True,
    ids=True,entities=True,entity_colors=True,entity_legend=False,
    show_deleted=False,raw_candidate_links=False,save_path=None):
    result=per_page_results[page_index]
    colors=final_entity_colors()
    records={row['text_index']:row for row in box_records if row['page_index']==page_index}
    page_groups=[row for row in grouped_records if row['page_index']==page_index]
    shown_entities=set()
    fig,ax=plt.subplots(figsize=(12,18));ax.imshow(crop_pages[page_index])

    def character_entity(index):return char_to_entity[f'PG{page_index+1}:C{index+1}']

    def text_visible(row):
        if row['delete_ocr'] and not show_deleted:return False
        return entity_filter=='All' or row.get('speaker')==entity_filter or (
            raw_candidate_links and entity_filter in row.get('entity_ids',[]))

    def role_color(row):
        if row.get('delete_ocr'):return 'red'
        if roles and row.get('text_role')=='NARRATED':return 'grey'
        entity=row.get('speaker')
        if entity_colors and entity in colors:return colors[entity]
        return 'gold' if row.get('text_role')=='SPOKEN' else 'deepskyblue'

    def mark(box,color,label='',style='-'):
        a,b,c,d=box
        ax.add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,edgecolor=color,linewidth=1.8,linestyle=style))
        if label:ax.text(a,max(0,b-3),label,fontsize=8,color='black',
            bbox=dict(facecolor=color,alpha=.82,edgecolor='none'))

    def text_label(row,index=None):
        parts=[]
        if ids:parts.append(row['box_id'])
        if row.get('delete_ocr'):parts.append('DELETED')
        elif roles:parts.append(row['text_role'])
        if entities and not row.get('delete_ocr'):
            parts.append(row.get('speaker') or row.get('speaker_status','unassociated'))
        if row.get('tail_override') and roles:parts.append('TAIL')
        if row.get('needs_review'):parts.append('review')
        if ocr:
            if index is None:
                text=row['text'] if ocr_source=='selected' else row.get(ocr_source+'_text','')
            else:
                decision=ocr_selection[page_index][index]
                text=decision['selected_text'] if ocr_source=='selected' else decision[ocr_source+'_text']
            parts.append(str(text or '')[:90])
        return ' | '.join(parts)

    if panels:
        for i,box in enumerate(result.get('panels',[])):mark(box,'limegreen',f'PN{i+1}' if ids else '')
    if characters:
        for i,box in enumerate(result['characters']):
            entity=character_entity(i)
            if entity_filter!='All' and entity!=entity_filter:continue
            shown_entities.add(entity)
            label=[]
            if ids:label.append(f'PG{PAGE_LABELS[page_index]}:C{i+1}')
            if entities:label.append(entity)
            mark(box,colors[entity] if entity_colors else 'dodgerblue',' / '.join(label))
    if texts:
        for index,box in enumerate(result['texts']):
            row=records[index]
            if not text_visible(row):continue
            if row.get('speaker') in colors:shown_entities.add(row['speaker'])
            mark(box,role_color(row),text_label(row,index),'--' if row['delete_ocr'] else '-')
    for group in page_groups:
        if entity_filter!='All' and group.get('speaker')!=entity_filter:continue
        color=role_color(group)
        if groups:mark(group['text_box'],color,text_label(group),'--')
        if contours:
            for segment in group['visible_segments']:
                points=np.asarray(segment);ax.plot(points[:,0],points[:,1],color=color,linewidth=2)
    if tails:
        for i,box in enumerate(result.get('tails',[])):mark(box,'purple',f'TAIL{i+1}' if ids else '')
    if same_entity:
        members={}
        for index,box in enumerate(result['characters']):
            entity=character_entity(index)
            if entity_filter=='All' or entity==entity_filter:members.setdefault(entity,[]).append(index)
        for entity,indices in members.items():
            if len(indices)<2:continue
            shown_entities.add(entity)
            a,b=center(result['characters'][indices[0]])
            for index in indices[1:]:
                c,d=center(result['characters'][index])
                ax.plot([a,c],[b,d],color=colors[entity] if entity_colors else 'blue',linewidth=1.5,alpha=.8)
    for enabled,key,target,style in [(text_character,'text_character_associations','characters','--'),
                                   (text_tail,'text_tail_associations','tails',':')]:
        if not enabled:continue
        for text_index,target_index in result.get(key,[]):
            row=records[text_index]
            if not text_visible(row):continue
            if row['delete_ocr'] and not raw_candidate_links:continue
            if target=='characters':
                entity=character_entity(target_index)
                if not raw_candidate_links and (row['text_role']!='SPOKEN' or row.get('speaker')!=entity):continue
                if entity_filter!='All' and entity!=entity_filter:continue
                color=colors[entity] if entity_colors else 'blue';shown_entities.add(entity)
                target_id=f'C{target_index+1}'
            else:
                color='purple';target_id=f'TAIL{target_index+1}'
            a,b=center(result['texts'][text_index]);c,d=center(result[target][target_index])
            ax.plot([a,c],[b,d],color=color,linestyle=style,linewidth=1.4,alpha=.85)
            if ids:ax.text((a+c)/2,(b+d)/2,f'T{text_index+1} -> {target_id}',fontsize=7)
    if entity_legend and shown_entities:
        handles=[Line2D([0],[0],color=colors[entity] if entity_colors else 'blue',lw=3,label=entity)
            for entity in colors if entity in shown_entities]
        ax.legend(handles=handles,title='Sequence entities',loc='upper left',bbox_to_anchor=(1,1))
    ax.set_title(f'{SEQUENCE_ID} | PG{PAGE_LABELS[page_index]} {PAGE_FILES[page_index].name} | {ocr_source} OCR')
    ax.axis('off');fig.tight_layout()
    if save_path:
        save_path=Path(save_path);save_path.parent.mkdir(parents=True,exist_ok=True)
        fig.savefig(save_path,bbox_inches='tight',dpi=150)
    buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=110)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))


def inspect_final_text(box_id):
    row=next(row for row in box_records if row['box_id']==box_id)
    crop,_=crop_box(crop_pages[row['page_index']],row['box'])
    if crop is not None:display(Image.fromarray(crop))
    print(json.dumps(row,ensure_ascii=False,indent=2))


# Close this viewer's previous callbacks on rerun. Earlier pipeline viewers stay independent.
if widgets is not None:
    old=globals().get('_full_final_viewer')
    if old:
        for control in old['controls'].values():control.unobserve_all();control.close()
        old['output'].close();old['panel'].close()
    clear_output(wait=True)
    final_page_picker=widgets.Dropdown(options=[(f'PG{PAGE_LABELS[i]} {p.name}',i) for i,p in enumerate(PAGE_FILES)],description='Page')
    final_ocr_picker=widgets.Dropdown(options=['selected','paddle','magi'],description='OCR')
    final_entity_picker=widgets.Dropdown(options=['All']+list(final_entity_colors()),description='Entity')
    defaults=dict(panels=False,texts=True,characters=True,tails=False,groups=False,contours=True,
        same_entity=False,text_character=True,text_tail=False,ocr=True,roles=True,ids=True,entities=True,
        entity_colors=True,entity_legend=False,show_deleted=False,raw_candidate_links=False)
    descriptions={'same_entity':'same entity lines','text_character':'text-character links',
        'text_tail':'text-tail links','roles':'spoken/narrated','entities':'entity numbers',
        'entity_colors':'entity colors','raw_candidate_links':'raw candidate links'}
    final_flags={key:widgets.Checkbox(value=value,description=descriptions.get(key,key.replace('_',' ')),indent=False)
        for key,value in defaults.items()}
    final_controls=dict(page_index=final_page_picker,ocr_source=final_ocr_picker,entity_filter=final_entity_picker,**final_flags)
    final_output=widgets.Output()
    def refresh_final_viewer(change=None):
        with final_output:
            clear_output(wait=True)
            show_final_page(**{key:control.value for key,control in final_controls.items()})
    for control in final_controls.values():control.observe(refresh_final_viewer,names='value')
    flag_list=list(final_flags.values())
    layout=[widgets.HBox([final_page_picker,final_ocr_picker,final_entity_picker])]
    layout += [widgets.HBox(flag_list[i:i+4]) for i in range(0,len(flag_list),4)]
    final_panel=widgets.VBox(layout+[final_output])
    _full_final_viewer=dict(controls=final_controls,output=final_output,panel=final_panel)
    display(final_panel);refresh_final_viewer()
else:
    print('Install ipywidgets for controls, or call show_final_page(0, same_entity=True).')
# Manual examples:
# show_final_page(0, same_entity=True, text_tail=True, entity_legend=True)
# inspect_final_text('PG1:T3')
